# Filrekommendationssystem

I denna uppgift byggs ett filrekommendationssystem med hjälp av MovieLens-datasetet.  
Målet är att rekommendera fem liknande filmer baserat på en given film.

Arbetet inleds med explorativ dataanalys (EDA), följt av en enkel baslinjemodell som enbart använder genrer.  
Modellen förbättras därefter genom att kombinera genrer och användargenererade taggar med hjälp av TF-IDF-vektorisering.  
Slutligen förbättras rekommendationskvaliteten ytterligare genom att filtrera bort filmer med för få betyg.

In [176]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

## Load the dataset

The main files used in this assignment are:
- movies.csv
- ratings.csv
- tags.csv

These files contain movie metadata, user ratings, and user-generated tags.

In [177]:
movies = pd.read_csv("C:/Users/samue/Documents/code/Machine-Learning-Samuel-Airisniemi/Labb1/movies.csv")
ratings = pd.read_csv("C:/Users/samue/Documents/code/Machine-Learning-Samuel-Airisniemi/Labb1/ratings.csv")
tags = pd.read_csv("C:/Users/samue/Documents/code/Machine-Learning-Samuel-Airisniemi/Labb1/tags.csv")

def fix_title(title):
    for article in ["The", "A", "An"]:
        if f", {article}" in title:
            return f"{article} " + title.replace(f", {article}", "")
    return title

movies["title_fixed"] = movies["title"].apply(fix_title)

print("movies: ", movies.shape)
print("ratings: ", ratings.shape)
print("tags: ", tags.shape)

movies:  (86537, 4)
ratings:  (33832162, 4)
tags:  (2328315, 4)


## Explorativ dataanalys

Innan rekommendationssystemet byggdes analyserades datasetet för att få en förståelse för dess storlek, struktur och vilka variabler som är användbara.

In [178]:
movies.head()

,movieId,title,genres,title_fixed
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995)
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995)
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995)


In [179]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   movieId      86537 non-null  int64 
 1   title        86537 non-null  object
 2   genres       86537 non-null  object
 3   title_fixed  86537 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.6+ MB


In [180]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


In [181]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33832162 entries, 0 to 33832161
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 1.0 GB


In [182]:
tags.head()

,userId,movieId,tag,timestamp
0,10,260,good vs evil,1430666558
1,10,260,Harrison Ford,1430666505
2,10,260,sci-fi,1430666538
3,14,1221,Al Pacino,1311600756
4,14,1221,mafia,1311600746


In [183]:
tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2328315 entries, 0 to 2328314
Data columns (total 4 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   userId     int64 
 1   movieId    int64 
 2   tag        object
 3   timestamp  int64 
dtypes: int64(3), object(1)
memory usage: 71.1+ MB


In [184]:
ratings.describe()

,userId,movieId,rating,timestamp
count,3.383216e+07,3.383216e+07,3.383216e+07,3.383216e+07
mean,1.654380e+05,2.831348e+04,3.542540e+00,1.269362e+09
std,9.534122e+04,4.992865e+04,1.063959e+00,2.541023e+08
min,1.000000e+00,1.000000e+00,5.000000e-01,7.896520e+08
25%,8.295300e+04,1.219000e+03,3.000000e+00,1.046718e+09
50%,1.661290e+05,3.263000e+03,4.000000e+00,1.264740e+09
75%,2.474500e+05,4.049100e+04,4.000000e+00,1.496919e+09
max,3.309750e+05,2.889830e+05,5.000000e+00,1.689843e+09


In [185]:
print("Number of movies: ", movies["movieId"].unique())
print("Number of users: ", ratings["userId"].unique())

Number of movies:  [     1      2      3 ... 288975 288977 288983]
Number of users:  [     1      2      3 ... 330973 330974 330975]


In [186]:
print(movies["genres"].head(10))

0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: object


In [187]:
tags.head(10)

,userId,movieId,tag,timestamp
0,10,260,good vs evil,1430666558
1,10,260,Harrison Ford,1430666505
2,10,260,sci-fi,1430666538
3,14,1221,Al Pacino,1311600756
4,14,1221,mafia,1311600746
5,14,58559,Atmospheric,1311530439
6,14,58559,Batman,1311530391
7,14,58559,comic book,1311530398
8,14,58559,dark,1311530428
9,14,58559,Heath Ledger,1311530404


In [188]:
tags["tag"].value_counts().head(10)

tag
sci-fi                14319
atmospheric           12172
action                10683
comedy                10161
surreal                9142
funny                  9094
visually appealing     8890
twist ending           8325
thought-provoking      7727
dark comedy            7659
Name: count, dtype: int64

In [189]:
print(movies.isnull().sum())
print(ratings.isnull().sum())
print(tags.isnull().sum())

movieId        0
title          0
genres         0
title_fixed    0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
userId        0
movieId       0
tag          17
timestamp     0
dtype: int64


In [190]:
ratings_per_movie = ratings.groupby("movieId").size()
ratings_per_movie.describe()

count     83239.000000
mean        406.446041
std        2806.975876
min           1.000000
25%           2.000000
50%           5.000000
75%          26.000000
max      122296.000000
dtype: float64

### Observationer från datan

- Datasetet innehåller ett stort antal filmer och betyg.
- Betygen är numeriska och representerar användarnas preferenser från låga till höga värden.
- Varje film kan tillhöra flera genrer.
- Användargenererade taggar ger mer detaljerade och specifika beskrivningar av filmer jämfört med enbart genrer.
- Många filmer har relativt få betyg, vilket kan påverka rekommendationskvaliteten.

## Baslinjemodell: genrebaserade rekommendationer

En enkel baslinjemodell skapades med hjälp av enbart filmers genrer.  
Varje films genrer omvandlades till en binär representation med flera etiketter (multi-label), och cosinuslikhet användes för att jämföra filmer.

In [191]:
movies["genres_list"] = movies["genres"].apply(lambda x: x.split("|"))
movies[["title", "genres", "genres_list"]].head()

,title,genres,genres_list
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,Jumanji (1995),Adventure|Children|Fantasy,"[Adventure, Children, Fantasy]"
2,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]"
3,Waiting to Exhale (1995),Comedy|Drama|Romance,"[Comedy, Drama, Romance]"
4,Father of the Bride Part II (1995),Comedy,[Comedy]


In [192]:
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(movies["genres_list"])

genres_df = pd.DataFrame(genres_encoded, columns = mlb.classes_)
genres_df.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [193]:
indices = pd.Series(movies.index, index = movies["title"]).drop_duplicates()
indices["GoldenEye (1995)"]

np.int64(9)

In [194]:
def recommend_movies(title):
    idx = indices[title]

    target_vector = genres_df.iloc[idx].values.reshape(1, -1)
    similarities = cosine_similarity(target_vector, genres_df)
    sim_scores = list(enumerate(similarities[0]))
    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)
    sim_scores = sim_scores[1:6]
    movie_indices = [i[0] for i in sim_scores]
    
    return movies["title_fixed"].iloc[movie_indices]

recommend_movies("Toy Story (1995)")

2203                                      Antz (1998)
3021                               Toy Story 2 (1999)
3654    The Adventures of Rocky and Bullwinkle (2000)
3913                  The Emperor's New Groove (2000)
4781                            Monsters, Inc. (2001)
Name: title_fixed, dtype: object

### Diskussion av baslinjemodellen

Denna baslinjemodell är enkel och lätt att implementera, men den har tydliga begränsningar.  
Genrer är breda kategorier, vilket innebär att många filmer kan framstå som liknande trots att de skiljer sig avsevärt i stil eller tema.

Modellen fångar likhet enbart baserat på genrer, vilket ofta leder till rekommendationer som är för generella och inte särskilt precisa när det gäller filmens faktiska innehåll eller stil.

### Minneshantering

Till en början testades att beräkna en fullständig cosinuslikhetsmatris mellan alla filmer.  
Detta krävde dock för mycket minne på grund av det stora antalet filmer i datasetet.

Istället för att beräkna alla parvisa likheter på en gång beräknades likheter endast vid behov för en vald film.  
Detta är betydligt mer minneseffektivt.

## Förbättrad modell: genrer och taggar med TF-IDF

För att förbättra baslinjemodellen kombinerades användargenererade taggar med filmers genrer.  
Genrerna behandlades som ytterligare nyckelord, och TF-IDF-vektorisering användes för att representera varje film som en textbaserad feature-vektor.

Denna metod gör det möjligt att fånga mer detaljerade likheter mellan filmer jämfört med att enbart använda genrer.

In [195]:
tags_grouped = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.astype(str))).reset_index()
tags_grouped.head()

,movieId,tag
0,1,animation friendship toys animation Disney Pix...
1,2,animals based on a book fantasy magic board ga...
2,3,sequel moldy old old age old men wedding old p...
3,4,characters chick flick girl movie characters c...
4,5,family pregnancy wedding 4th wall aging baby d...


In [196]:
movies_with_tags = movies.merge(tags_grouped, on = "movieId", how = "left")
movies_with_tags.head()

,movieId,title,genres,title_fixed,genres_list,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",animation friendship toys animation Disney Pix...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995),"[Adventure, Children, Fantasy]",animals based on a book fantasy magic board ga...
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995),"[Comedy, Romance]",sequel moldy old old age old men wedding old p...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",characters chick flick girl movie characters c...
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995),[Comedy],family pregnancy wedding 4th wall aging baby d...


In [197]:
movies_with_tags["tag"] = movies_with_tags["tag"].fillna("")
movies_with_tags["genres_text"] = movies_with_tags["genres"].str.replace("|", " ", regex = False)
movies_with_tags.head()

,movieId,title,genres,title_fixed,genres_list,tag,genres_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",animation friendship toys animation Disney Pix...,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995),"[Adventure, Children, Fantasy]",animals based on a book fantasy magic board ga...,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995),"[Comedy, Romance]",sequel moldy old old age old men wedding old p...,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",characters chick flick girl movie characters c...,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995),[Comedy],family pregnancy wedding 4th wall aging baby d...,Comedy


In [198]:
movies_with_tags["combined_features"] = (movies_with_tags["genres_text"] + " " + movies_with_tags["tag"])
movies_with_tags[["title", "genres_text", "tag", "combined_features"]].head(10)

,title,genres_text,tag,combined_features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy,animation friendship toys animation Disney Pix...,Adventure Animation Children Comedy Fantasy an...
1,Jumanji (1995),Adventure Children Fantasy,animals based on a book fantasy magic board ga...,Adventure Children Fantasy animals based on a ...
2,Grumpier Old Men (1995),Comedy Romance,sequel moldy old old age old men wedding old p...,Comedy Romance sequel moldy old old age old me...
3,Waiting to Exhale (1995),Comedy Drama Romance,characters chick flick girl movie characters c...,Comedy Drama Romance characters chick flick gi...
4,Father of the Bride Part II (1995),Comedy,family pregnancy wedding 4th wall aging baby d...,Comedy family pregnancy wedding 4th wall aging...
5,Heat (1995),Action Crime Thriller,Al Pacino complex characters crime philosophy ...,Action Crime Thriller Al Pacino complex charac...
6,Sabrina (1995),Comedy Romance,based on a play Harrison Ford Paris romance si...,Comedy Romance based on a play Harrison Ford P...
7,Tom and Huck (1995),Adventure Children,adapted from:book author:Mark Twain prospect p...,Adventure Children adapted from:book author:Ma...
8,Sudden Death (1995),Action,Jean-Claude Van Damme Can't remember CLV 1990s...,Action Jean-Claude Van Damme Can't remember CL...
9,GoldenEye (1995),Action Adventure Thriller,ITS AN OK MOVIE IF YOU LIKE JAMES BOUND. 007 ...,Action Adventure Thriller ITS AN OK MOVIE IF Y...


In [199]:
tfidf = TfidfVectorizer(stop_words = "english")
tfidf_matrix = tfidf.fit_transform(movies_with_tags["combined_features"])
print(tfidf_matrix.shape)

(86537, 51797)


In [200]:
indices_tfidf = pd.Series(movies_with_tags.index, index = movies_with_tags["title"]).drop_duplicates()

In [201]:
def recommend_movies_tfidf(title, top_n = 5):
    matches = movies_with_tags[movies_with_tags["title"].str.contains(title, case = False, na = False)]
    
    if matches.empty:
        return "Ingen film hittades."
    
    idx = matches.index[0]
    
    target_vector = tfidf_matrix[idx]
    similarities = cosine_similarity(target_vector, tfidf_matrix).flatten()
    
    sim_scores = list(enumerate(similarities))
    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)
    
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    return movies_with_tags[["title_fixed", "genres"]].iloc[movie_indices]

recommend_movies_tfidf("Toy Story")

,title_fixed,genres
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2264,A Bug's Life (1998),Adventure|Animation|Children|Comedy
14815,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
39883,Finding Dory (2016),Adventure|Animation|Comedy
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy


In [202]:
recommend_movies_tfidf("Matrix")

,title_fixed,genres
6248,The Matrix Reloaded (2003),Action|Adventure|Sci-Fi|Thriller|IMAX
6810,The Matrix Revolutions (2003),Action|Adventure|Sci-Fi|Thriller|IMAX
2015,Tron (1982),Action|Adventure|Sci-Fi
6668,Avalon (2001),Drama|Fantasy|Sci-Fi
1678,Dark City (1998),Adventure|Film-Noir|Sci-Fi|Thriller


In [203]:
recommend_movies_tfidf("Pulp Fiction")

,title_fixed,genres
1062,Reservoir Dogs (1992),Crime|Mystery|Thriller
1663,Jackie Brown (1997),Crime|Drama|Thriller
17,Four Rooms (1995),Comedy
7300,Kill Bill: Vol. 2 (2004),Action|Drama|Thriller
6752,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller


In [204]:
recommend_movies_tfidf("Forrest Gump")

,title_fixed,genres
3054,The Green Mile (1999),Crime|Drama
7849,The Terminal (2004),Comedy|Drama|Romance
2705,Big (1988),Comedy|Drama|Fantasy|Romance
503,Philadelphia (1993),Drama
5196,Joe Versus the Volcano (1990),Comedy|Romance


### Jämförelse mellan modeller

Vid jämförelse mellan baslinjemodellen (endast genrer) och TF-IDF-modellen:

- Baslinjemodellen tenderar att rekommendera filmer som delar breda kategorier.
- TF-IDF-modellen fångar mer specifika likheter genom användargenererade taggar.

Exempel:

Input: Toy Story

Baslinjerekommendationer:
- Antz (1998)
- Toy Story 2 (1999)
- The Adventures of Rocky and Bullwinkle (2000)
- The Emperor's New Groove (2000)
- Monsters, Inc. (2001)

TF-IDF-rekommendationer:
- Toy Story 2 (1999)
- A Bug's Life (1998)
- Toy Story 3 (2010)
- Finding Dory (2016)
- Monsters, Inc. (2001)

TF-IDF-modellen ger mer semantiskt relevanta rekommendationer.

TF-IDF ger högre vikt till ovanliga och mer beskrivande taggar, vilket gör att modellen kan särskilja filmer mer precist jämfört med en representation som enbart bygger på genrer.

## Filtrering baserat på antal betyg

En ytterligare förbättring var att filtrera de rekommenderade filmerna genom att kräva ett minsta antal betyg.  
Detta minskar risken att rekommendera filmer med mycket begränsad användarfeedback.

In [205]:
rating_counts = ratings.groupby("movieId").size().reset_index(name = "rating_count")
rating_counts.head()

,movieId,rating_count
0,1,76813
1,2,30209
2,3,15820
3,4,3028
4,5,15801


In [206]:
movies_with_tags = movies_with_tags.merge(rating_counts, on = "movieId", how = "left")
movies_with_tags["rating_count"] = movies_with_tags["rating_count"].fillna(0)
movies_with_tags[["title", "rating_count"]].head()

,title,rating_count
0,Toy Story (1995),76813.0
1,Jumanji (1995),30209.0
2,Grumpier Old Men (1995),15820.0
3,Waiting to Exhale (1995),3028.0
4,Father of the Bride Part II (1995),15801.0


In [207]:
movies_with_tags["rating_count"].describe()

count     86537.000000
mean        390.956030
std        2754.067218
min           0.000000
25%           2.000000
50%           5.000000
75%          24.000000
max      122296.000000
Name: rating_count, dtype: float64

In [208]:
def recommend_movies_tfidf_filtered(title, top_n = 5, min_ratings = 50):
    matches = movies_with_tags[movies_with_tags["title"].str.contains(title, case = False, na = False)]
    
    if matches.empty:
        return "Ingen film hittades."
    
    idx = matches.index[0]
    
    target_vector = tfidf_matrix[idx]
    similarities = cosine_similarity(target_vector, tfidf_matrix).flatten()
    
    sim_df = pd.DataFrame({"movie_index": movies_with_tags.index, "similarity": similarities})
    sim_df = sim_df[sim_df["movie_index"] != idx]
    sim_df = sim_df.merge(movies_with_tags[["title", "title_fixed", "genres", "rating_count"]], left_on = "movie_index", right_index = True)
    sim_df = sim_df[sim_df["rating_count"] >= min_ratings]
    sim_df = sim_df.sort_values("similarity", ascending = False)
    
    return sim_df[["title_fixed", "genres", "rating_count", "similarity"]].head(top_n)

recommend_movies_tfidf_filtered("lord of the rings")

,title_fixed,genres,rating_count,similarity
5841,The Lord of the Rings: The Two Towers (2002),Adventure|Fantasy,73687.0,0.580832
7029,The Lord of the Rings: The Return of the King ...,Action|Adventure|Drama|Fantasy,75512.0,0.572802
4888,The Lord of the Rings: The Fellowship of the R...,Adventure|Fantasy,79940.0,0.550854
23627,The Hobbit: The Battle of the Five Armies (2014),Adventure|Fantasy,8785.0,0.537717
20613,The Hobbit: The Desolation of Smaug (2013),Adventure|Fantasy|IMAX,12475.0,0.480053


### Varför detta hjälper

Vissa filmer kan matcha den valda filmen väl utifrån textbaserade features, men har väldigt få betyg och kan därför vara alltför okända eller osäkra rekommendationer.  
Genom att filtrera baserat på antal betyg förbättras den praktiska kvaliteten och trovärdigheten i rekommendationerna. Denna förbättring minskar även problemet med gles eller opålitlig data, där filmer med mycket få betyg annars kan framstå som liknande enbart på grund av begränsad tagginformation.

## Example recommendations

Below are some example recommendations produced by the system.

In [209]:
recommend_movies_tfidf_filtered("toy story")

,title_fixed,genres,rating_count,similarity
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,34495.0,0.900875
2264,A Bug's Life (1998),Adventure|Animation|Children|Comedy,28290.0,0.813194
14815,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,21131.0,0.743159
39883,Finding Dory (2016),Adventure|Animation|Comedy,5706.0,0.729255
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,48441.0,0.718466


In [210]:
recommend_movies_tfidf_filtered("matrix")

,title_fixed,genres,rating_count,similarity
6248,The Matrix Reloaded (2003),Action|Adventure|Sci-Fi|Thriller|IMAX,30788.0,0.862561
6810,The Matrix Revolutions (2003),Action|Adventure|Sci-Fi|Thriller|IMAX,24470.0,0.814648
2015,Tron (1982),Action|Adventure|Sci-Fi,13172.0,0.556981
6668,Avalon (2001),Drama|Fantasy|Sci-Fi,761.0,0.554078
1678,Dark City (1998),Adventure|Film-Noir|Sci-Fi|Thriller,15086.0,0.540994


In [211]:
recommend_movies_tfidf_filtered("pulp fiction")

,title_fixed,genres,rating_count,similarity
1062,Reservoir Dogs (1992),Crime|Mystery|Thriller,45318.0,0.849265
1663,Jackie Brown (1997),Crime|Drama|Thriller,15878.0,0.790045
17,Four Rooms (1995),Comedy,6688.0,0.721911
7300,Kill Bill: Vol. 2 (2004),Action|Drama|Thriller,38553.0,0.695925
6752,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller,46973.0,0.683680


## Slutsats

I denna uppgift byggdes ett filrekommendationssystem med hjälp av MovieLens-datasetet.

En enkel baslinjemodell baserad på genrer implementerades först.  
Därefter förbättrades modellen genom att kombinera genrer och användargenererade taggar med hjälp av TF-IDF-vektorisering.  
Slutligen förbättrades rekommendationskvaliteten ytterligare genom att filtrera bort filmer med för få betyg.

Resultaten visade att den TF-IDF-baserade modellen gav mer detaljerade och relevanta rekommendationer än baslinjemodellen som enbart använde genrer.  
En begränsning är att systemet fortfarande är innehållsbaserat och inte fullt ut utnyttjar kollaborativ filtrering baserat på användar–objekt-interaktioner.

Möjliga framtida förbättringar inkluderar:
- kollaborativ filtrering
- hybrida rekommendationsmetoder
- bättre hantering av dubbletter eller tvetydiga filmtitlar

### Begränsningar

- Modellen är starkt beroende av kvaliteten och tillgängligheten av taggar
- Vissa filmer har begränsad data, vilket påverkar rekommendationskvaliteten
- Systemet är inte personligt och ger samma rekommendationer till alla användare
- Användningen av partiell textmatchning kan leda till felaktiga träffar om flera filmer har liknande titlar